# Step6 — Índice de Persistencia (Ecuador, 2015–2025)

**Proyecto:** HCI Ecuador (Hazard Chronicity Index)
**Fase:** 6 — Índice de Persistencia, por provincia y por cantón
**Autor:** Jerson Achicanoy (FAO OER) · implementación asistida
**Fecha:** 2026-08-05

**Objetivo de este notebook:** construir el Índice de Persistencia (Fase 0
§9: "meses consecutivos afectados, duración, longitud de temporada"), con
tres componentes:

1. **Racha máxima** — la racha más larga de meses consecutivos con al
   menos un evento.
2. **Duración media** — el largo promedio de todas las rachas de meses
   consecutivos afectados (no solo la más larga).
3. **Longitud de temporada** — en cuántos meses distintos del año
   calendario (1 a 12) ha ocurrido al menos un evento alguna vez, sobre
   toda la ventana 2015-2025 — mide qué tan concentrada o dispersa está la
   afectación a lo largo del año.

**Alcance temporal confirmado por Jerson (AskUserQuestion, 2026-08-05):**
toda la línea de tiempo 2015-2025 (132 meses), **no** restringido a los 38
meses "evento ENSO" de la Fase 5 — mide cronicidad general del cantón
(temporada invernal completa, lluvias prolongadas), no solo lo
estrictamente atribuible a El Niño. Ver justificación completa en
`docs/methodology/06_persistence_index.md`.

**English:** Persistence Index

Measures how long events last once they start: longest run of consecutive affected months, typical run length and number of calendar months affected per canton and province, combined into a 0-1 index.

Outputs: `persistence_index_*` tables in `data/processed/`.

## 0. Configuración del entorno

**English:** 0. Setup

In [ ]:
# Fase 6 tampoco requiere GEE ni CDS.

# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent  # notebook vive en <repo>/notebooks/
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Tabular data handling
import pandas as pd  # tablas
# Numerical operations
import numpy as np  # operaciones numéricas
# Efficient reader for GeoPackage layers
import pyogrio  # lectura eficiente de hazard_event_geo.gpkg

# Show up to 40 columns when printing DataFrames
pd.set_option("display.max_columns", 40)

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)

# Create a logger that writes to the logs folder
logger = get_logger("step6_persistence_index", config, log_dir=PROJECT_ROOT / "logs")
logger.info("Fase 6 iniciada — configuración cargada desde %s", CONFIG_PATH)

# First year of the analysis window
YEAR_MIN = config["date_range"]["start_year"]  # 2015
# Last year of the analysis window
YEAR_MAX = config["date_range"]["end_year"]  # 2025
# List of all years in the analysis window
ALL_YEARS = list(range(YEAR_MIN, YEAR_MAX + 1))
# The twelve calendar months
ALL_MONTHS = list(range(1, 13))
# Los 132 pares (año, mes) completos de la ventana del proyecto — mismo
# principio de reindexado explícito que Fase 3 (Mann-Kendall) y Fase 5
# (validación de hipótesis): un mes sin eventos cuenta como 0/False, no
# desaparece de la serie.
# Complete timeline of (year, month) pairs in the window
TIMELINE = [(y, m) for y in ALL_YEARS for m in ALL_MONTHS]
# Check the timeline has 132 months (11 years x 12)
assert len(TIMELINE) == 132

# Persistence index settings from the config
PER_CFG = config["persistence_index"]
# Weights of the three persistence components
COMPONENT_WEIGHTS = PER_CFG["component_weights"]
# Number of calendar months used to scale season length
SEASON_DENOM = PER_CFG["season_length_denominator"]  # 12
# Check the component weights add up to 1
assert abs(sum(COMPONENT_WEIGHTS.values()) - 1.0) < 1e-3, "los pesos de componentes deben sumar 1.0"
# Log the component weights
logger.info("Pesos de componentes de Persistencia: %s", COMPONENT_WEIGHTS)

## 1. Carga de datos

Se parte de `hazard_event_geo.gpkg` (Fase 2), igual que las fases
anteriores, con el mismo join de atributo sobre `canton_code` para los
nombres oficiales (evita repetir el bug de Fase 3 §4.1).

**English:** 1. Load data

In [ ]:
# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Load the hazard events table without geometry
he = pyogrio.read_dataframe(processed_dir / "hazard_event_geo.gpkg", read_geometry=False)
# Log the number of events loaded
logger.info("hazard_event_geo cargado: %d filas", len(he))

# Load the official canton and province names
adm2 = pyogrio.read_dataframe(
    processed_dir / "admin_boundaries.gpkg", layer="adm2_canton", read_geometry=False
)[["canton_code", "canton_name_official", "province_name_official"]].drop_duplicates()
# Remember the row count to detect accidental changes
n_antes = len(he)
# Attach official canton and province names to each event
he = he.drop(columns=["province_name_official"]).merge(adm2, on="canton_code", how="left")
# Check the join did not change the number of events
assert len(he) == n_antes
# Check every event received an official name
assert he[["canton_name_official", "province_name_official"]].isna().sum().sum() == 0
# Log that official names were attached by canton code
logger.info("Nombres oficiales incorporados por atributo (canton_code); cobertura 100%%")

# Columns that identify a canton
CANTON_COLS = ["province_code", "province_name_official", "canton_code", "canton_name_official"]
# Columns that identify a province
PROVINCE_COLS = ["province_code", "province_name_official"]

## 2. Rachas de meses consecutivos afectados

Función central: dada la serie booleana de 132 meses (afectado / no
afectado) de una unidad administrativa, identifica cada racha de meses
consecutivos `True` y devuelve su longitud. Técnica estándar: un cambio de
valor entre un mes y el anterior marca el inicio de una racha nueva; se
agrupa por el número acumulado de cambios (`cumsum` de la señal de cambio).

**English:** 2. Runs of consecutive affected months

In [ ]:
# Lengths of runs of consecutive affected months
def calcular_rachas(afectado: pd.Series) -> list[int]:
    """Devuelve la longitud de cada racha de meses consecutivos con
    `afectado == True` en una serie booleana ordenada temporalmente.

    Args:
        afectado: serie booleana, una fila por mes, ya ordenada por
            (year, month) y reindexada a la ventana completa (132 meses),
            sin huecos.

    Returns:
        Lista con la longitud de cada racha de True consecutivos. Lista
        vacía si la unidad nunca tuvo un mes afectado.

    English:
    Lengths of runs of consecutive affected months.

    Args:
        afectado: Boolean series, True for months with events.

    Returns:
        List with the length of each affected run.
    """
    # Id that changes each time the affected state flips
    cambio = (afectado != afectado.shift(fill_value=False)).cumsum()  # nuevo grupo en cada cambio de valor
    # Sum and size of each run
    rachas = afectado.groupby(cambio).agg(["sum", "size"])  # sum=True si el grupo es de afectados
    # Keep only the runs where the unit was affected
    rachas_afectadas = rachas[afectado.groupby(cambio).first()]  # solo grupos donde afectado=True
    # Return the length of each affected run
    return rachas_afectadas["size"].tolist()


# Compute the three persistence components for one unit
def calcular_persistencia_por_unidad(eventos_mes: set[tuple[int, int]]) -> tuple[int, float, int]:
    """Calcula los tres componentes de persistencia para una unidad administrativa.

    Args:
        eventos_mes: conjunto de pares (year, month) en que la unidad tuvo
            al menos un evento (de la vista correspondiente: un hazard_type
            o el total combinado).

    Returns:
        (racha_maxima, duracion_media, longitud_temporada):
        - racha_maxima: longitud de la racha de meses consecutivos más larga.
        - duracion_media: promedio de todas las rachas (0 si no hay ninguna).
        - longitud_temporada: número de meses calendario distintos (1-12)
          con al menos un evento alguna vez en la ventana.

    English:
    Compute the three persistence components for one administrative unit.

    Args:
        eventos_mes: Set of (year, month) pairs with at least one event.

    Returns:
        Longest run, mean run length and season length (distinct calendar months affected).
    """
    # Boolean series: was the unit affected in each month of the timeline
    afectado = pd.Series([par in eventos_mes for par in TIMELINE])
    # Lengths of all affected runs
    rachas = calcular_rachas(afectado)
    # Longest run, zero if there are none
    racha_maxima = max(rachas) if rachas else 0
    # Mean run length, zero if there are none
    duracion_media = (sum(rachas) / len(rachas)) if rachas else 0.0
    # Calendar months (1-12) in which the unit was ever affected
    meses_calendario = {m for (y, m) in eventos_mes}
    # Season length: number of distinct calendar months affected
    longitud_temporada = len(meses_calendario)
    # Return the three components
    return racha_maxima, duracion_media, longitud_temporada

## 3. Cálculo por cantón y provincia, desglosado por peligro y combinado

Mismo patrón de las Fases 3-5: por `hazard_type` y con una fila sintética
`"TODOS"`. Para cada unidad administrativa y vista, se arma el conjunto de
meses con evento y se aplican los tres componentes de la sección 2.

**English:** 3. Canton and province level, per hazard and combined

In [ ]:
# Persistence components for every unit of one administrative level
def ejecutar_nivel_persistencia(df: pd.DataFrame, admin_cols: list[str]) -> pd.DataFrame:
    """Calcula los tres componentes de persistencia por unidad administrativa,
    desglosado por hazard_type y combinado ('TODOS').

    Args:
        df: hazard_event_geo con nombres oficiales ya incorporados.
        admin_cols: columnas del nivel administrativo (cantón o provincia).

    Returns:
        DataFrame con una fila por (unidad administrativa, hazard_type),
        incluida la vista 'TODOS', con racha_maxima/duracion_media/
        longitud_temporada — cobertura completa del universo de unidades
        (mismo principio de Fase 5 §4.6: unidades sin eventos en una vista
        dada igual aparecen, con los tres componentes en 0).

    English:
    Compute persistence components for every unit of one administrative level, per hazard and combined.

    Args:
        df: Event table with official names.
        admin_cols: Columns that define the administrative level.

    Returns:
        Table with one row per unit and hazard view, including units without events (components at zero).
    """
    # Universe of all units that have any historical event
    universo = df[admin_cols].drop_duplicates()

    # Compute the components for one hazard view
    def _para_hazard(sub_df: pd.DataFrame, hazard_label: str) -> pd.DataFrame:
        # List that collects one row per unit
        filas = []
        # Loop over every administrative unit
        for _, unidad in universo.iterrows():
            # Select the events that belong to the unit
            mask = (sub_df[admin_cols] == unidad).all(axis=1)
            # Distinct (year, month) pairs with events for the unit
            eventos_mes = set(map(tuple, sub_df.loc[mask, ["year", "month"]].drop_duplicates().values))
            # Compute the three components for the unit
            racha_max, dur_media, temp_len = calcular_persistencia_por_unidad(eventos_mes)
            # Store the row with the components and the event count
            filas.append({**unidad.to_dict(), "hazard_type": hazard_label,
                          "racha_maxima": racha_max, "duracion_media": round(dur_media, 3),
                          "longitud_temporada": temp_len, "n_eventos": mask.sum()})
        # Return the table for this view
        return pd.DataFrame(filas)

    # One table per hazard type
    partes = [_para_hazard(df[df.hazard_type == h], h) for h in sorted(df.hazard_type.unique())]
    # Add the combined view of all hazards
    partes.append(_para_hazard(df, "TODOS"))
    # Stack all views
    return pd.concat(partes, ignore_index=True)

In [ ]:
# Log the start of the canton-level calculation
logger.info("Calculando componentes de Persistencia a nivel cantón (puede tardar unos segundos)...")
# Compute persistence at canton level
per_canton = ejecutar_nivel_persistencia(he, CANTON_COLS)
# Log the start of the province-level calculation
logger.info("Calculando componentes de Persistencia a nivel provincia...")
# Compute persistence at province level
per_provincia = ejecutar_nivel_persistencia(he, PROVINCE_COLS)
# Log the number of rows produced
logger.info("Cantón: %d filas | Provincia: %d filas", len(per_canton), len(per_provincia))

## 4. Normalización y cálculo del índice compuesto

**Racha máxima** y **duración media** se normalizan con min-max, igual que
en la Fase 4 (Severidad) — no tienen un techo natural interpretable (132
meses seguidos afectados no es una referencia realista), así que se
comparan contra el máximo observado en la propia vista.

**Longitud de temporada** se normaliza dividiendo entre 12 directamente
(`season_length_denominator` en `config.yaml`) — sí tiene un techo natural
y alcanzable: una unidad con eventos en los 12 meses calendario en algún
punto de 11 años obtiene 1.0, sin necesidad de comparar contra otras
unidades.

Los tres componentes normalizados se combinan en partes iguales (pesos en
`config.yaml`, `persistence_index.component_weights`) — Fase 0 §9 no da una
fórmula ponderada para Persistencia (a diferencia de Severidad), así que
partes iguales es el valor por defecto más neutral, pendiente del mismo
análisis de sensibilidad ya señalado para las demás fases.

**English:** 4. Normalization and composite index

In [ ]:
# Normalize the components and combine them into the index
def normalizar_y_combinar(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza los tres componentes de persistencia y calcula el índice compuesto.

    Args:
        df: tabla de persistencia cruda (racha_maxima, duracion_media,
            longitud_temporada), con columna 'hazard_type' para normalizar
            cada vista por separado.

    Returns:
        Copia de df con norm_racha_maxima, norm_duracion_media,
        norm_longitud_temporada y persistence_index, todos en [0,1].

    English:
    Normalize the three components and combine them into the persistence index.

    Args:
        df: Table with longest run, mean duration and season length.

    Returns:
        Copy of the table with the normalized columns and 'persistence_index' in [0, 1].
    """
    # Work on a copy
    df = df.copy()

    # Min-max scaling of one series
    def _minmax(s: pd.Series) -> pd.Series:
        # Range of the series
        rango = s.max() - s.min()
        # A constant series has no range
        if rango == 0:
            # Return zeros for a constant series
            return pd.Series(0.0, index=s.index)
        # Scale to the 0-1 interval
        return (s - s.min()) / rango

    # Normalize the longest run within each hazard view
    df["norm_racha_maxima"] = df.groupby("hazard_type")["racha_maxima"].transform(_minmax)
    # Normalize the mean duration within each hazard view
    df["norm_duracion_media"] = df.groupby("hazard_type")["duracion_media"].transform(_minmax)
    # Scale season length by the calendar-month denominator, capped at 1
    df["norm_longitud_temporada"] = (df["longitud_temporada"] / SEASON_DENOM).clip(upper=1.0)

    # Weighted sum of the three normalized components
    df["persistence_index"] = (
        COMPONENT_WEIGHTS["racha_maxima"] * df["norm_racha_maxima"]
        + COMPONENT_WEIGHTS["duracion_media"] * df["norm_duracion_media"]
        + COMPONENT_WEIGHTS["longitud_temporada"] * df["norm_longitud_temporada"]
    )
    # Check the index is between 0 and 1
    assert df["persistence_index"].between(0, 1).all(), "persistence_index fuera de [0,1]"
    # Return the table with the index
    return df

In [ ]:
# Persistence index at canton level
per_canton = normalizar_y_combinar(per_canton)
# Persistence index at province level
per_provincia = normalizar_y_combinar(per_provincia)
# Log that the index is valid at both levels
logger.info("Índice de Persistencia calculado y verificado en [0,1] para ambos niveles")

## 5. Validación

**English:** 5. Validation

In [ ]:
# Chequeo 1: cobertura completa en cada vista — 221 cantones x 5 vistas (4 hazard_type + TODOS).
# Number of hazard types in the data
n_hazard_types = he.hazard_type.nunique()
# Check every canton appears in every view, including the combined one
assert len(per_canton) == 221 * (n_hazard_types + 1), "cobertura incompleta a nivel cantón"
# Check every province appears in every view
assert len(per_provincia) == 24 * (n_hazard_types + 1), "cobertura incompleta a nivel provincia"

# Chequeo 2: la racha máxima nunca puede superar 132 (el total de meses de la ventana).
# Check no canton run exceeds the 132-month timeline
assert (per_canton["racha_maxima"] <= 132).all()
# Check no province run exceeds the timeline
assert (per_provincia["racha_maxima"] <= 132).all()

# Chequeo 3: longitud_temporada nunca puede superar 12 (meses calendario posibles).
# Check season length never exceeds 12 months
assert (per_canton["longitud_temporada"] <= 12).all()
# Check season length never exceeds 12 months at province level
assert (per_provincia["longitud_temporada"] <= 12).all()

# Chequeo 4: duracion_media nunca puede superar racha_maxima (la racha más larga
# es, por definición, al menos tan larga como el promedio de todas las rachas).
# Check the mean duration never exceeds the longest run
assert (per_canton["duracion_media"] <= per_canton["racha_maxima"] + 1e-9).all()

# Ten cantons with the highest persistence
top10 = per_canton[per_canton.hazard_type == "TODOS"].sort_values("persistence_index", ascending=False).head(10)
# Print the top ten
print(top10[["province_name_official", "canton_name_official", "racha_maxima",
             "duracion_media", "longitud_temporada", "persistence_index"]].to_string(index=False))

## 6. Exportación

**English:** 6. Export

In [ ]:
# Tables to export
tablas = {
    "persistence_index_canton": per_canton,
    "persistence_index_provincia": per_provincia,
}

# Loop over the tables
for nombre, df in tablas.items():
    # Save as Parquet
    df.to_parquet(processed_dir / f"{nombre}.parquet", index=False)
    # Save as CSV
    df.to_csv(processed_dir / f"{nombre}.csv", index=False)
    # Log each exported table
    logger.info("Exportado: %s (%d filas)", nombre, len(df))

# Print the export folder
print(f"Tablas exportadas a {processed_dir} :")
# Loop over the exported tables
for nombre in tablas:
    # Print each table name
    print(f" - {nombre}")

## 7. Próximos pasos

- **Fase 7 — Índice de Magnitud ENSO**: con ICEN ya confirmado como ancla
  (Fase 5 §4.2), traducir Débil/Moderado/Fuerte/Extraordinario a valores
  0-1.
- **Fase 8 — HCI compuesto**: combinar Severidad (Fase 4), Recurrencia
  (Fase 5), Persistencia (esta fase) y Magnitud ENSO (Fase 7). Pesos y
  parámetros pendientes de análisis de sensibilidad: los de esta fase
  (partes iguales 1/3, §4), los de Severidad (Fase 4 §6, incluidos los dos
  valores atípicos ya identificados) y el umbral de "evento ENSO" (Fase 5
  §10, posible redefinición basada en episodios consecutivos en vez de mes
  a mes).